# NOVA Gold Lab — XAUUSD H1 CPU Research
Reproducible, research-only CPU training for Exness XAUUSDm H1. No broker credentials, orders, or live-trading actions.

**Dataset contract:** requires timestamp, OHLC, and genuine MT5 `tick_volume` (or a documented `volume` column). The currently committed PR #238 CSV contains OHLC only; it will be rejected until genuine volume is supplied. Never synthesize or fill volume.

**Attach one verified dataset** using Kaggle Add Input. This notebook trains XGBoost and a small PyTorch LSTM only after schema and data-quality checks. Wave labels are never fabricated.

In [ ]:
from pathlib import Path
import os, json, hashlib, random, warnings
import numpy as np
import pandas as pd
SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_FILE = ""  # Optional exact path, e.g. /kaggle/input/<dataset>/<file>.csv
OUTPUT_DIR = Path('/kaggle/working/nova_training_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_LSTM = True
HORIZON = 5
SEQUENCE_LENGTH = 32
LSTM_EPOCHS = 8
print('CPU-only mode:', True, '| output:', OUTPUT_DIR)

In [ ]:
def validate_ohlcv(frame):
    """Normalize and validate genuine OHLCV data; never invent missing volume."""
    data = frame.copy()
    data.columns = [str(column).strip().lower() for column in data.columns]
    aliases = {
        'timestamp_utc': 'timestamp', 'datetime': 'timestamp',
        'date': 'timestamp', 'time': 'timestamp',
        'tick_volume': 'volume',
    }
    rename = {old: new for old, new in aliases.items()
              if old in data.columns and new not in data.columns}
    data = data.rename(columns=rename)
    required = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(
            f"Missing required OHLCV columns {missing}. "
            "XAUUSD H1 source must provide genuine MT5 tick_volume; "
            "volume must not be fabricated."
        )
    raw_timestamp = data['timestamp']
    numeric_timestamp = pd.to_numeric(raw_timestamp, errors='coerce')
    if numeric_timestamp.notna().all():
        unit = 'ms' if float(numeric_timestamp.abs().median()) > 1e11 else 's'
        data['timestamp'] = pd.to_datetime(
            numeric_timestamp, unit=unit, utc=True, errors='coerce'
        )
    else:
        data['timestamp'] = pd.to_datetime(
            raw_timestamp, utc=True, errors='coerce'
        )
    if data['timestamp'].isna().any():
        raise ValueError('Invalid or missing timestamps found')
    for column in ['open', 'high', 'low', 'close', 'volume']:
        data[column] = pd.to_numeric(data[column], errors='coerce')
    if data[required].isna().any().any():
        raise ValueError('Missing or non-numeric OHLCV values found')
    if data['timestamp'].duplicated().any():
        raise ValueError('Duplicate timestamps found; resolve before training')
    invalid = (
        (data.high < data.low) |
        (data.open < data.low) | (data.open > data.high) |
        (data.close < data.low) | (data.close > data.high) |
        (data.volume < 0)
    )
    if invalid.any():
        raise ValueError('Invalid OHLC bounds or negative volume')
    return data.sort_values('timestamp').reset_index(drop=True)


In [ ]:
candidates = sorted(Path('/kaggle/input').rglob('*.csv')) if Path('/kaggle/input').exists() else []
if DATA_FILE:
    data_path = Path(DATA_FILE)
elif len(candidates) == 1:
    data_path = candidates[0]
elif len(candidates) > 1:
    hourly = [p for p in candidates if any(tag in p.stem.lower() for tag in ('1h', 'h1', 'hour'))]
    if len(hourly) == 1:
        data_path = hourly[0]
    else:
        raise ValueError('Multiple CSV files found and no unique H1 file. Set DATA_FILE to one exact file: ' + '\\n'.join(map(str,candidates[:30])))
elif len(candidates) == 0:
    raise FileNotFoundError('No CSV found under /kaggle/input. Attach a Kaggle dataset using Add Input.')
else:
    raise ValueError('Multiple CSV files found. Set DATA_FILE to one exact file: ' + '\\n'.join(map(str,candidates[:30])))
if not data_path.is_file(): raise FileNotFoundError(data_path)
df = validate_ohlcv(pd.read_csv(data_path, sep=None, engine='python'))
if len(df) < 1000: raise ValueError(f'Need at least 1,000 valid bars for this baseline; got {len(df)}')
print('Dataset:', data_path, '| rows:', len(df), '| range:', df.timestamp.min(), '→', df.timestamp.max())

In [ ]:
def rsi(s, period=14):
    delta=s.diff(); gain=delta.clip(lower=0).rolling(period).mean(); loss=(-delta.clip(upper=0)).rolling(period).mean()
    rs=gain/loss.replace(0, np.nan); return 100-(100/(1+rs))
feat = pd.DataFrame(index=df.index)
feat['return_1']=df.close.pct_change()
feat['return_5']=df.close.pct_change(5)
feat['rsi_14']=rsi(df.close)
ema12=df.close.ewm(span=12, adjust=False).mean(); ema26=df.close.ewm(span=26, adjust=False).mean()
feat['macd']=ema12-ema26; feat['macd_signal']=feat.macd.ewm(span=9, adjust=False).mean()
feat['volatility_20']=feat.return_1.rolling(20).std()
feat['volume_ratio']=df.volume/(df.volume.rolling(20).mean().replace(0,np.nan))
feat['candle_range']=(df.high-df.low)/df.close.replace(0,np.nan)
future_return=df.close.shift(-HORIZON)/df.close-1
labels=(future_return>0).astype('int8').rename('label')
valid=feat.replace([np.inf,-np.inf],np.nan).notna().all(axis=1) & future_return.notna()
X=feat.loc[valid].reset_index(drop=True); y=labels.loc[valid].reset_index(drop=True)
timestamps=df.loc[valid,'timestamp'].reset_index(drop=True)
n=len(X); train_end=int(n*0.70); val_end=int(n*0.85)
# Purge HORIZON rows at split boundaries so training labels never look into the next partition.
train_limit=train_end-HORIZON; val_limit=val_end-HORIZON
if min(train_limit, val_limit-train_end, n-val_end) < 100: raise ValueError('Not enough rows after purging horizon overlap at split boundaries')
if y.iloc[:train_limit].nunique() < 2: raise ValueError('Training partition contains only one class; inspect dataset/label horizon')
print('Samples:', n, '| train:',train_limit,'| validation:',val_limit-train_end,'| test:',n-val_end,'| purged bars per boundary:',HORIZON)

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
def evaluate(name, y_true, prob):
    pred=(np.asarray(prob)>=0.5).astype(int)
    out={'model':name,'accuracy':float(accuracy_score(y_true,pred)),'balanced_accuracy':float(balanced_accuracy_score(y_true,pred)),
         'precision':float(precision_score(y_true,pred,zero_division=0)),'recall':float(recall_score(y_true,pred,zero_division=0)),
         'f1':float(f1_score(y_true,pred,zero_division=0))}
    out['roc_auc']=float(roc_auc_score(y_true,prob)) if len(np.unique(y_true))==2 else None
    return out
X_train,X_val,X_test=X.iloc[:train_limit],X.iloc[train_end:val_limit],X.iloc[val_end:]
y_train,y_val,y_test=y.iloc[:train_limit],y.iloc[train_end:val_limit],y.iloc[val_end:]
dummy=DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
baseline=evaluate('majority_baseline',y_test,dummy.predict_proba(X_test)[:,1])
xgb=XGBClassifier(n_estimators=350,max_depth=4,learning_rate=0.03,subsample=0.8,colsample_bytree=0.8,
                  objective='binary:logistic',eval_metric='logloss',n_jobs=2,tree_method='hist',random_state=SEED)
xgb.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=False)
xgb_val=evaluate('xgboost_validation',y_val,xgb.predict_proba(X_val)[:,1])
xgb_test=evaluate('xgboost_test',y_test,xgb.predict_proba(X_test)[:,1])
print(json.dumps({'baseline':baseline,'validation':xgb_val,'test':xgb_test},indent=2))

In [ ]:
lstm_test=None; lstm_val=None
if RUN_LSTM:
    import torch
    from torch import nn
    from sklearn.preprocessing import StandardScaler
    torch.manual_seed(SEED); torch.set_num_threads(2); device=torch.device('cpu')
    scaler=StandardScaler().fit(X_train)
    xs=scaler.transform(X).astype('float32'); ys=y.to_numpy(dtype='float32')
    def sequences(start,end):
        idx=np.arange(max(start,SEQUENCE_LENGTH-1),end)
        xx=np.stack([xs[i-SEQUENCE_LENGTH+1:i+1] for i in idx])
        return torch.tensor(xx),torch.tensor(ys[idx,None])
    tx,ty=sequences(SEQUENCE_LENGTH-1,train_limit); vx,vy=sequences(train_end,val_limit); qx,qy=sequences(val_end,n)
    class SmallLSTM(nn.Module):
        def __init__(self,n_features):
            super().__init__(); self.lstm=nn.LSTM(n_features,32,batch_first=True); self.head=nn.Sequential(nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
        def forward(self,z):
            h,_=self.lstm(z); return self.head(h[:,-1,:])
    net=SmallLSTM(xs.shape[1]).to(device); opt=torch.optim.Adam(net.parameters(),lr=0.001); loss_fn=nn.BCEWithLogitsLoss()
    best_state=None; best_loss=float('inf')
    for epoch in range(LSTM_EPOCHS):
        net.train(); order=torch.randperm(len(tx))
        for ids in order.split(256):
            opt.zero_grad(); loss=loss_fn(net(tx[ids]),ty[ids]); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): val_loss=float(loss_fn(net(vx),vy).item())
        if val_loss<best_loss: best_loss=val_loss; best_state={k:v.detach().clone() for k,v in net.state_dict().items()}
        print(f'epoch {epoch+1}/{LSTM_EPOCHS} validation_loss={val_loss:.5f}')
    net.load_state_dict(best_state); net.eval()
    with torch.no_grad():
        lstm_val=evaluate('lstm_validation',vy.numpy().ravel(),torch.sigmoid(net(vx)).numpy().ravel())
        lstm_test=evaluate('lstm_test',qy.numpy().ravel(),torch.sigmoid(net(qx)).numpy().ravel())
    torch.save({'state_dict':net.state_dict(),'feature_names':list(X.columns),'scaler_mean':scaler.mean_.tolist(),'scaler_scale':scaler.scale_.tolist(),'sequence_length':SEQUENCE_LENGTH},OUTPUT_DIR/'lstm_cpu.pt')
    print(json.dumps({'lstm_validation':lstm_val,'lstm_test':lstm_test},indent=2))

In [ ]:
import platform, sys
xgb.save_model(OUTPUT_DIR/'xgboost_cpu.json')
pd.DataFrame({'feature':X.columns,'importance':xgb.feature_importances_}).sort_values('importance',ascending=False).to_csv(OUTPUT_DIR/'xgboost_feature_importance.csv',index=False)
metrics={'baseline_test':baseline,'xgboost_validation':xgb_val,'xgboost_test':xgb_test,'lstm_validation':lstm_val,'lstm_test':lstm_test,
         'samples':n,'train_rows':train_limit,'validation_rows':val_limit-train_end,'test_rows':n-val_end,'purged_horizon_rows_per_boundary':HORIZON,'horizon_bars':HORIZON,
         'test_start':str(timestamps.iloc[val_end]),'test_end':str(timestamps.iloc[-1]),'feature_names':list(X.columns),
         'cpu_only':True,'live_trading_enabled':False,'orders_submitted':False,'promotion_approved':False}
(OUTPUT_DIR/'metrics.json').write_text(json.dumps(metrics,indent=2))
hasher=hashlib.sha256()
with data_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024*1024), b''): hasher.update(chunk)
sha=hasher.hexdigest()
manifest={'dataset_file':data_path.name,'dataset_sha256':sha,'dataset_bytes':data_path.stat().st_size,'python':sys.version,
          'platform':platform.platform(),'random_seed':SEED,'run_mode':'kaggle-free-cpu','outputs':sorted(p.name for p in OUTPUT_DIR.iterdir()),
          'note':'Research candidate only. No model auto-promotion, broker writes, or live trading.'}
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Saved outputs:', sorted(p.name for p in OUTPUT_DIR.iterdir()))
print('Compare XGBoost test accuracy against majority baseline:', xgb_test['accuracy'], 'vs', baseline['accuracy'])
print('Do not promote unless out-of-sample, walk-forward, costs, trade-count and risk gates all pass.')